<a href="https://colab.research.google.com/github/Dheena2411/CRM-CM-CW1/blob/task1_data_prep/CRM%26CM_CW1_T1%262.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Task 1- Data Preparation & Feature Engineering

In [15]:
!pip install -q yfinance lxml

import sys, pandas as pd, numpy as np, yfinance as yf
import sklearn, scipy, matplotlib
import os, time

print("Python", sys.version.split()[0])
for m in [pd, np, yf, sklearn, scipy, matplotlib]:
    print(m.__name__, m.__version__)

Python 3.13.18
pandas 2.2.3
numpy 2.4.6
yfinance 0.2.66
sklearn 1.6.1
scipy 1.16.3
matplotlib 3.10.0


In [16]:
START_DATE = "2021-01-01"   # inclusive
END_DATE   = "2025-01-01"   # exclusive, so the last trading day is 31 Dec 2024
BENCHMARK  = "^GSPC"
RETRIEVAL_DATE = pd.Timestamp.today().strftime("%Y-%m-%d")  # record this in the report
print("Data retrieved on:", RETRIEVAL_DATE)

Data retrieved on: 2026-11-19


Task 1.1

In [22]:
# Part 1: S&P 500 constituents from Wikipedia
# Adapted from the Week tutorial (SP500_RFM_Analysis_2022_2024): adds caching so the
# retrieval date and constituent list stay fixed when the notebook is re-run.
import os, json

TICKER_FILE = "sp500_tickers.csv"
WIKI_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

if os.path.exists(TICKER_FILE):
    sp500 = pd.read_csv(TICKER_FILE)
    RETRIEVAL_DATE = sp500["retrieval_date"].iloc[0]
    print("Loaded saved ticker list retrieved on", RETRIEVAL_DATE)
else:
    sp500 = pd.read_html(WIKI_URL, match="Symbol",
                         storage_options={"User-Agent": "Mozilla/5.0"})[0]
    sp500["yf_ticker"] = sp500["Symbol"].str.replace(".", "-", regex=False)
    sp500["retrieval_date"] = RETRIEVAL_DATE
    sp500.to_csv(TICKER_FILE, index=False)

assert sp500["yf_ticker"].is_unique, "Duplicate tickers found"
print("Total tickers extracted:", len(sp500))
sp500[["Symbol", "yf_ticker", "Security", "GICS Sector"]].head(10)

Loaded saved ticker list retrieved on 2026-11-19
Total tickers extracted: 503


,Symbol,yf_ticker,Security,GICS Sector
0,MMM,MMM,3M,Industrials
1,AOS,AOS,A. O. Smith,Industrials
2,ABT,ABT,Abbott Laboratories,Health Care
3,ABBV,ABBV,AbbVie,Health Care
4,ACN,ACN,Accenture,Information Technology
5,ADBE,ADBE,Adobe Inc.,Information Technology
6,AMD,AMD,Advanced Micro Devices,Information Technology
7,AES,AES,AES Corporation,Utilities
8,AFL,AFL,Aflac,Financials
9,A,A,Agilent Technologies,Health Care


In [23]:
import time

DATA_FILE = "sp500_daily_2021_2024.parquet"
META_FILE = "download_metadata.json"
tickers = sp500["yf_ticker"].tolist() + [BENCHMARK]

def download_batch(batch, retries=3):
    for attempt in range(retries):
        try:
            return yf.download(batch, start=START_DATE, end=END_DATE,
                               auto_adjust=False, actions=True,
                               group_by="column", threads=True, progress=False)
        except Exception as e:
            print(f"  Attempt {attempt+1} failed: {e}")
            time.sleep(5)
    raise RuntimeError(f"Batch failed after {retries} attempts: {batch[:3]}...")

if os.path.exists(DATA_FILE):
    prices = pd.read_parquet(DATA_FILE)
    print("Loaded saved snapshot:", DATA_FILE)
else:
    frames = []
    for i in range(0, len(tickers), 50):
        batch = tickers[i:i+50]
        long = download_batch(batch).stack(level=1, future_stack=True)
        long.index.names = ["Date", "Ticker"]
        frames.append(long.reset_index())
        print(f"Batch {i//50 + 1} done ({len(batch)} tickers)")
        time.sleep(2)
    prices = pd.concat(frames, ignore_index=True)
    prices.columns.name = None
    prices.to_parquet(DATA_FILE, index=False)

    # Provenance record
    meta = {
        "ticker_source": WIKI_URL,
        "ticker_retrieval_date": RETRIEVAL_DATE,
        "price_source": "Yahoo Finance via yfinance",
        "yfinance_version": yf.__version__,
        "pandas_version": pd.__version__,
        "download_timestamp": pd.Timestamp.now().isoformat(timespec="seconds"),
        "start_date": START_DATE, "end_date_exclusive": END_DATE,
        "benchmark": BENCHMARK, "auto_adjust": False,
        "n_tickers_requested": len(tickers),
    }
    with open(META_FILE, "w") as f:
        json.dump(meta, f, indent=2)

prices.head()

Loaded saved snapshot: sp500_daily_2021_2024.parquet


,Date,Ticker,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume
0,2021-01-04,A,114.009132,118.639999,0.194,120.089996,117.110001,118.940002,0.0,2030700.0
1,2021-01-04,AAPL,125.537140,129.410004,0.000,133.610001,126.760002,133.520004,0.0,143301900.0
2,2021-01-04,ABBV,84.219139,105.410004,0.000,107.349998,103.860001,107.180000,0.0,9523400.0
3,2021-01-04,ABNB,139.149994,139.149994,0.000,151.005005,137.000000,150.990005,0.0,6409900.0
4,2021-01-04,ABT,97.269707,109.110001,0.000,110.250000,107.260002,109.529999,0.0,6192000.0


In [24]:
got = prices.dropna(subset=["Adj Close"])["Ticker"].unique()
failed = sorted(set(tickers) - set(got))

print("Total rows:", len(prices))
print("Tickers with data:", len(got), "of", len(tickers))
print("Date range:", prices["Date"].min().date(), "to", prices["Date"].max().date())
print("Failed downloads:", failed)

Total rows: 506520
Tickers with data: 499 of 504
Date range: 2021-01-04 to 2024-12-31
Failed downloads: ['FDXF', 'HONA', 'Q', 'SNDK', 'VYLR']


In [25]:
print(round(os.path.getsize(DATA_FILE) / 1e6, 1), "MB")
from google.colab import files
files.download(DATA_FILE)
files.download("sp500_tickers.csv")

11.9 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>